In [4]:
!pip install -q ray xgboost imbalanced-learn scikit-learn

import os
import time
import numpy as np
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve, auc
from sklearn.impute import SimpleImputer
from tqdm import tqdm
import ray

# 1. Inicialização limpa e programática do Ray (sem comandos de terminal travados)
ray.shutdown()
# Forçando a inicialização de um novo cluster local para evitar erros de conexão prévios
ray.init(ignore_reinit_error=True, num_cpus=os.cpu_count())

print("===== RAY INICIALIZADO COM SUCESSO =====")
print(f"Recursos disponíveis: {ray.cluster_resources()}")

# 2. Geração e Preparação do Dataset (Reduzido para treinamento ultrarápido)
print("\nGerando dataset sintético...")
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

N_MODELOS = 10
N_ESTIMATORS = 40  # Dobrado de 20 para 40 para deixar o treinamento 2x mais demorado que o anterior

# 3. Função de Treinamento Individual
def treinar_xgb(i, X_tr, y_tr):
    smote = SMOTE(random_state=i, sampling_strategy='auto', k_neighbors=3)
    X_res, y_res = smote.fit_resample(X_tr, y_tr)

    model = XGBClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=i,
        eval_metric='logloss',
        tree_method='hist',
        device='cpu',
        n_jobs=1
    )
    model.fit(X_res, y_res)
    return model

# 4. Função de Avaliação do Ensemble
def avaliar(modelos):
    def prever_ensemble(X_data):
        todas_preds = np.array([model.predict(X_data) for model in modelos])
        return (np.mean(todas_preds, axis=0) > 0.5).astype(int)

    def prever_probabilidades_ensemble(X_data):
        todas_probs = np.array([model.predict_proba(X_data)[:, 1] for model in modelos])
        return np.mean(todas_probs, axis=0)

    y_pred = prever_ensemble(X_test)
    y_probs = prever_probabilidades_ensemble(X_test)

    precision, recall, _ = precision_recall_curve(y_test, y_probs)
    pr_auc_score = auc(recall, precision)

    print(f"\nF1 Score: {f1_score(y_test, y_pred):.4f}")
    print(f"PR AUC: {pr_auc_score:.4f}")
    print(f"Precisão Classe 1: {precision_score(y_test, y_pred, pos_label=1):.2f}")
    print(f"Recall Classe 1: {recall_score(y_test, y_pred, pos_label=1):.2f}")
    print("-" * 30)

# 5. Execução Distribuída no Ray com Barra de Progresso
print(f"\n===== DISTRIBUINDO {N_MODELOS} MODELOS NO RAY =====")
t1 = time.time()

# Armazena os dados no Object Store distribuído do Ray
X_train_ref = ray.put(X_train)
y_train_ref = ray.put(y_train)

@ray.remote
def treinar_xgb_ray(i):
    X_tr = ray.get(X_train_ref)
    y_tr = ray.get(y_train_ref)
    print(f"[Worker] Iniciando modelo {i+1}...", flush=True)
    model = treinar_xgb(i, X_tr, y_tr)
    print(f"[Worker] Modelo {i+1} concluído! ✅", flush=True)
    return model

futures_ray = [treinar_xgb_ray.remote(i) for i in range(N_MODELOS)]
modelos_ray = []

remaining = futures_ray
with tqdm(total=N_MODELOS, desc="Treinando Ensemble no Ray") as pbar:
    while remaining:
        done, remaining = ray.wait(remaining, num_returns=1)
        for f in done:
            modelos_ray.append(ray.get(f))
            pbar.update(1)

tempo_ray = time.time() - t1
print(f"\nTempo total de execução no Ray: {tempo_ray:.2f}s")

# 6. Avaliação Final
avaliar(modelos_ray)

# 7. Limpeza do Ray
ray.shutdown()
print("Sessão do Ray encerrada com sucesso.")

2026-10-18 23:22:04,289	INFO worker.py:2033 -- Started a local Ray instance.


===== RAY INICIALIZADO COM SUCESSO =====
Recursos disponíveis: {'object_store_memory': 3982516224.0, 'CPU': 2.0, 'node:172.28.0.12': 1.0, 'node:__internal_head__': 1.0, 'memory': 9292537856.0}

Gerando dataset sintético...

===== DISTRIBUINDO 10 MODELOS NO RAY =====


Treinando Ensemble no Ray:   0%|          | 0/10 [00:00<?, ?it/s]

(treinar_xgb_ray pid=5137) [Worker] Iniciando modelo 1...


Treinando Ensemble no Ray:  10%|█         | 1/10 [00:03<00:34,  3.79s/it]

(treinar_xgb_ray pid=5137) [Worker] Modelo 1 concluído! ✅


Treinando Ensemble no Ray: 100%|██████████| 10/10 [00:04<00:00,  2.09it/s]



Tempo total de execução no Ray: 4.83s

F1 Score: 0.9062
PR AUC: 0.9767
Precisão Classe 1: 0.95
Recall Classe 1: 0.87
------------------------------
(treinar_xgb_ray pid=5136) [Worker] Iniciando modelo 10... [repeated 9x across cluster]
(treinar_xgb_ray pid=5137) [Worker] Modelo 9 concluído! ✅ [repeated 8x across cluster]
Sessão do Ray encerrada com sucesso.
